In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# غيّر المسار ده للمسار الصحيح عندك
MERGED_MODEL_PATH = "/content/drive/MyDrive/CHATMODEL/mergedModel"          # أو "/content/drive/MyDrive/merged_qwen_medical"

print("جاري تحميل الـ tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_PATH)

print("جاري تحميل الموديل المددمج...")
model = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_PATH,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

model.eval()
print("التحميل خلّص ✓")
print(f"عدد الـ parameters القابلة للتدريب: {model.num_parameters():,}")

جاري تحميل الـ tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


جاري تحميل الموديل المددمج...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

التحميل خلّص ✓
عدد الـ parameters القابلة للتدريب: 1,543,714,304


In [ ]:
# نفس الـ SYSTEM اللي كنت بتستخدمه
SYSTEM_PROMPT = """أنت مساعد طبي للمساعدة الأولية فقط داخل تطبيق صحي.
افهم أي لغة، لكن رد بالعربية دائمًا.
قواعد صارمة:
1) لا علاج/لا أدوية/لا جرعات.
2) لا تشخيص نهائي.
3) اذكر 2 إلى 4 أمراض محتملة فقط.
4) إذا المعلومات غير كافية: اسأل سؤالين توضيحيين محددين.
5) إذا ظهرت مؤشرات خطيرة: وجّه للطوارئ فورًا.
صيغة الرد:
🔎 تحليل الأعراض:
🦠 أمراض محتملة:
❓ معلومات إضافية قد تساعد:
⚠️ تنبيه:
"""

def test_inference(symptoms):
    user_text = f"الأعراض: {symptoms}\n\nالمطلوب: اختر 2-4 أمراض محتملة فقط، اذكر سبب مختصر لكل واحدة، واسأل سؤالين إذا احتجت."

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=280,
            do_sample=False,
            repetition_penalty=1.2,
            temperature=0.1,
            top_p=0.85,
           # repetition_penalty=1.1,
           # do_sample=True
        )

    response = tokenizer.decode(outputs[0][len(inputs["input_ids"][0]):], skip_special_tokens=True)
    return response.strip()

# اختبار
print(test_inference("صداع شديد، حساسية للضوء، غثيان وقيء"))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


🔎 تحليل الأعراض:
الأعراض المذكورة قد تشير إلى أكثر من حالة طبية.

🦠 أمراض محتملة:
1- migraine: يتوافق مع مجموعة من الأعراض المذكورة.
2- tension headache: يتوافق مع مجموعة من الأعراض المذكورة.
3- brain cancer: يتوافق مع مجموعة من الأعراض المذكورة.
4- fibromyalgia: يتوافق مع مجموعة من الأعراض المذكورة.
5- subdural hemorrhage: يتوافق مع مجموعة من الأعراض المذكورة.
6- intracranial abscess: يتوافق مع مجموعة من الأعراض المذكورة.
7- concussion: يتوافق مع مجموعة من الأعراض المذكورة.
8- chronic pain disorder: يتوافق مع مجموعة من الأعراض المذكورة.
9- neuralgia (neck): يتوافق مع مجموعة من الأعراض المذكورة.
10- spondylosis: يتوافق مع مجموعة من الأعراض المذكورة.
11- degenerative disc disease: يتوافق مع مجموعة من الأعراض المذكورة.
12- sickle cell crisis: يتوافق مع مجموعة من الأعراض المذكورة.
13- autonomic nervous system disorder: يتوافق مع مجموعة
